# Medical RAG Bot - Evaluation with LangSmith

We test the chatbot the same way you would test any software: a fixed set of questions
with known answers, run through the bot, scored automatically, with results saved in LangSmith.

**The workflow**
1. **Dataset**: questions taken from our own PDFs, each with a reference answer and the PDF it comes from
2. **Target**: the RAG chain we want to test (the same one `app.py` runs)
3. **Evaluators**: functions that give each answer a score
4. **Experiment**: `client.evaluate()` runs every question through the target, scores it, and uploads the results

**Everything here is free.** The chatbot and the "LLM-as-a-judge" both use Qwen through Groq's free API.
Because the judge is the same model that wrote the answers, it may go easy on its own style
(self-preference bias). So we also use two **code-based** evaluators that don't need an LLM at all.
Read the judge's comments in LangSmith before you trust a score.

> Groq's free tier allows only a limited number of tokens per minute. One run is about 65 API calls
> (13 answers + 52 judge calls), so it's normal for it to pause and retry. Let it finish.

## 1. Setup

In [52]:
import os

if os.path.basename(os.getcwd()) == "research":
    os.chdir("..")

from dotenv import load_dotenv
load_dotenv()

for key in ("PINECONE_API_KEY", "LANGSMITH_API_KEY", "GROQ_API_KEY", "GROQ_MODEL"):
    assert os.getenv(key), f"{key} missing in .env"

GROQ_MODEL = os.getenv("GROQ_MODEL")
print("Working directory:", os.getcwd())
print("LLM (chatbot + judge):", GROQ_MODEL)

Working directory: d:\Downloads\RAG-Based-Medical-ChatBot-main
LLM (chatbot + judge): openai/gpt-oss-120b


## 2. Load the chatbot (the *target* we are testing)

In [53]:
from src.helper import download_hugging_face_embeddings, load_llm
from src.rag import load_vector_store, build_rag_chain

embeddings = download_hugging_face_embeddings()
vector_store = load_vector_store(embeddings)
llm = load_llm()   # Qwen via the Groq API

K = 3   # number of chunks the retriever returns
rag_chain = build_rag_chain(vector_store, llm, k=K)

Device set to use cuda:0


In [54]:
def file_name(path):
    """'data\\Diabetes.pdf' (Windows) or 'data/Diabetes.pdf' (Linux) -> 'Diabetes.pdf'"""
    return path.replace("\\", "/").split("/")[-1]


def target(inputs: dict) -> dict:
    """LangSmith calls this once per question in the dataset."""
    result = rag_chain.invoke({"input": inputs["question"]})
    docs = result["context"]
    return {
        "answer": result["answer"].strip(),
        "contexts": [doc.page_content for doc in docs],                    # text of the retrieved chunks
        "sources": [file_name(doc.metadata.get("source", "")) for doc in docs],  # which PDF each came from
    }


# Try it once before running the whole evaluation
test = target({"question": "What is Charcot's triad?"})
print("Answer :", test["answer"])
print("Sources:", test["sources"])

Answer : Charcot's triad refers to the combination of jaundice, upper abdominal pain, and fever as symptoms associated with acute cholangitis.
Sources: ['First Principles of Gastroenterology.pdf', 'First Principles of Gastroenterology.pdf', 'First Principles of Gastroenterology.pdf']


## 3. Create the dataset for groundness check

In [55]:
from langsmith import Client

client = Client()
DATASET_NAME = "medical-bot-eval-v1"

OUT_OF_SCOPE = "I don't know. The medical documents do not contain this information."

examples = [
    # ---------- Diabetes.pdf ----------
    {
        "inputs": {"question": "What is diabetes?"},
        "outputs": {
            "answer": "Diabetes is a chronic health problem characterized by high blood glucose levels "
                      "resulting from defects in insulin production, insulin action, or both.",
            "source": "Diabetes.pdf",
        },
    },
    {
        "inputs": {"question": "How many people in the United States had type 2 diabetes in 2007?"},
        "outputs": {
            "answer": "In 2007, 23.6 million people, or 7.8% of the United States population, had type 2 diabetes.",
            "source": "Diabetes.pdf",
        },
    },
    # ---------- Allergen Immunotherapy.pdf ----------
    {
        "inputs": {"question": "What color codes are used for allergen immunotherapy vials?"},
        "outputs": {
            "answer": "Red is the maintenance concentrate (1:1 vol/vol, #1), yellow is the 10-fold dilution "
                      "(1:10, #2), blue is the 100-fold dilution (1:100, #3), green is the 1000-fold dilution "
                      "(1:1000, #4) and silver is the 10,000-fold dilution (1:10,000, #5).",
            "source": "Allergen Immunotherapy.pdf",
        },
    },
    {
        "inputs": {"question": "How long is the expiration dating period for 50% glycerin bulk allergen extracts?"},
        "outputs": {
            "answer": "A total of six years from the time of extraction: a maximum of three years for "
                      "manufacturer storage and three years for shipped vial expiration dating.",
            "source": "Allergen Immunotherapy.pdf",
        },
    },
    # ---------- STANDARD TREATMENT.pdf ----------
    {
        "inputs": {"question": "How should P. vivax malaria be treated?"},
        "outputs": {
            "answer": "With chloroquine for three days and primaquine for 14 days. Primaquine prevents relapse "
                      "but is contraindicated in pregnant women, infants and people with G6PD deficiency.",
            "source": "STANDARD TREATMENT.pdf",
        },
    },
    {
        "inputs": {"question": "What are the types of poisonous snake bite?"},
        "outputs": {
            "answer": "Poisonous snake bites are of two types: neurotoxic and vasculotoxic.",
            "source": "STANDARD TREATMENT.pdf",
        },
    },
    # ---------- First Principles of Gastroenterology.pdf ----------
    {
        "inputs": {"question": "What is Charcot's triad?"},
        "outputs": {
            "answer": "Intermittent fever and chills, jaundice and abdominal pain. It is the classical "
                      "presentation of acute bacterial (ascending) cholangitis.",
            "source": "First Principles of Gastroenterology.pdf",
        },
    },
    {
        "inputs": {"question": "How is H. pylori infection treated?"},
        "outputs": {
            "answer": "With triple therapy: a proton pump inhibitor (PPI) plus two appropriate antibiotics taken "
                      "twice daily for 7-14 days. A urea breath test after the patient has been off the PPI "
                      "for 2 weeks confirms eradication.",
            "source": "First Principles of Gastroenterology.pdf",
        },
    },
    {
        "inputs": {"question": "What does a high serum-ascites albumin gradient (SAAG) indicate?"},
        "outputs": {
            "answer": "A SAAG above 11 g/L means the ascites is due to portal hypertension (for example cirrhosis). "
                      "A low gradient means it is not from portal hypertension; malignancy is the likeliest cause.",
            "source": "First Principles of Gastroenterology.pdf",
        },
    },
    # ---------- clinical_medicine_ashok_chandra.pdf ----------
    {
        "inputs": {"question": "What does a complete clinical diagnosis include?"},
        "outputs": {
            "answer": "The site of disease (anatomical), the nature of disease (etiological), the "
                      "pathophysiology (complications) and the functional disturbance (dysfunction).",
            "source": "clinical_medicine_ashok_chandra.pdf",
        },
    },
    {
        "inputs": {"question": "What are the common causes of cardiac disease?"},
        "outputs": {
            "answer": "Congenital, rheumatic, hypertensive, infectious (e.g. tuberculous or viral pericarditis, "
                      "viral myocarditis, subacute bacterial endocarditis) and atherosclerotic heart disease.",
            "source": "clinical_medicine_ashok_chandra.pdf",
        },
    },
    # ---------- Out of scope: the bot should say it doesn't know ----------
    {
        "inputs": {"question": "Who won the 2011 Cricket World Cup?"},
        "outputs": {"answer": OUT_OF_SCOPE, "source": None},
    },
    {
        "inputs": {"question": "How do I reset my Wi-Fi router?"},
        "outputs": {"answer": OUT_OF_SCOPE, "source": None},
    },
]

if client.has_dataset(dataset_name=DATASET_NAME):
    print(f"Dataset '{DATASET_NAME}' already exists - reusing it.")
else:
    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description="Questions from the 5 medical PDFs + 2 out-of-scope questions",
    )
    client.create_examples(dataset_id=dataset.id, examples=examples)
    print(f"Created dataset '{DATASET_NAME}' with {len(examples)} examples.")

Dataset 'medical-bot-eval-v1' already exists - reusing it.


## 4. Evaluators

An evaluator is a function that receives the question (`inputs`), the bot's output (`outputs`) and,
if it needs one, the reference (`reference_outputs`), and returns a score.
LangSmith works out which arguments to pass from the **parameter names**, so keep these names exactly.

Each evaluator returns `{"key": name, "score": number, "comment": why}`. The comment shows up in LangSmith.

| Evaluator | Compares | Uses an LLM? | Question it answers |
|---|---|---|---|
| `source_hit` | retrieved sources vs reference source | No | Did retrieval find the right PDF? |
| `answer_similarity` | answer vs reference answer | No (embeddings) | How close in meaning is the answer? (0-1) |
| `correctness` | answer vs reference answer | Yes | Is the answer factually right? |
| `groundedness` | answer vs retrieved chunks | Yes | Did the bot stick to the documents, or make things up? |
| `relevance` | answer vs question | Yes | Does the answer address the question? |
| `retrieval_relevance` | retrieved chunks vs question | Yes | Were the retrieved chunks about the question? |

### 4.1 Code-based evaluators (no LLM, so they are fast and give the same result every run)

In [56]:
import numpy as np


def source_hit(outputs: dict, reference_outputs: dict) -> dict:
    """1 if any retrieved chunk came from the PDF that contains the answer, else 0."""
    expected = reference_outputs["source"]
    if expected is None:   # out-of-scope question: there is no correct PDF
        return {"key": "source_hit", "score": None, "comment": "Out-of-scope question, not scored"}

    hit = expected in outputs["sources"]
    return {
        "key": "source_hit",
        "score": int(hit),
        "comment": f"expected {expected} | retrieved {outputs['sources']}",
    }


def answer_similarity(outputs: dict, reference_outputs: dict) -> dict:
    """Cosine similarity between the embeddings of the bot answer and the reference answer."""
    a = np.array(embeddings.embed_query(outputs["answer"]))
    b = np.array(embeddings.embed_query(reference_outputs["answer"]))
    cosine = float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
    return {"key": "answer_similarity", "score": round(cosine, 3)}

### 4.2 LLM-as-a-judge evaluators

We send the judge clear instructions and ask it to **start its reply with YES or NO**, then give one
sentence of reasoning. `YES` becomes score 1 and `NO` becomes score 0. The full reply is kept as the comment, so
you can see *why* it decided.

In [57]:
judge_llm = llm


def ask_judge(instructions: str, content: str):
    """Send one grading request to the judge and turn its YES/NO reply into a 1/0 score."""
    reply = judge_llm.invoke([("system", instructions), ("human", content)]).content.strip()
    first_word = reply.lstrip("*#> ").upper()
    score = 1 if first_word.startswith("YES") else 0
    return score, reply


def join_chunks(contexts: list) -> str:
    return "\n\n".join(f"[Chunk {i + 1}] {text}" for i, text in enumerate(contexts))

In [58]:
CORRECTNESS_INSTRUCTIONS = """You are a teacher grading a quiz.
You will be given a QUESTION, the CORRECT ANSWER and a STUDENT ANSWER.
Grade the STUDENT ANSWER only on whether its facts agree with the CORRECT ANSWER.
Extra information is fine as long as it does not contradict the CORRECT ANSWER.
If the CORRECT ANSWER says "I don't know", the STUDENT ANSWER is correct only if it also says it doesn't know.

Start your reply with YES (correct) or NO (incorrect), then give one short sentence explaining why."""


def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    content = (
        f"QUESTION: {inputs['question']}\n"
        f"CORRECT ANSWER: {reference_outputs['answer']}\n"
        f"STUDENT ANSWER: {outputs['answer']}"
    )
    score, reply = ask_judge(CORRECTNESS_INSTRUCTIONS, content)
    return {"key": "correctness", "score": score, "comment": reply}

In [59]:
GROUNDEDNESS_INSTRUCTIONS = """You are a teacher grading a quiz.
You will be given FACTS and a STUDENT ANSWER.
The STUDENT ANSWER is grounded if everything it says is supported by the FACTS.
It is NOT grounded if it contains information that is not in the FACTS (made-up information).
An answer that only says "I don't know" is grounded.

Start your reply with YES (grounded) or NO (not grounded), then give one short sentence explaining why."""


def groundedness(outputs: dict) -> dict:
    content = f"FACTS:\n{join_chunks(outputs['contexts'])}\n\nSTUDENT ANSWER: {outputs['answer']}"
    score, reply = ask_judge(GROUNDEDNESS_INSTRUCTIONS, content)
    return {"key": "groundedness", "score": score, "comment": reply}

In [60]:
RELEVANCE_INSTRUCTIONS = """You are a teacher grading a quiz.
You will be given a QUESTION and a STUDENT ANSWER.
The STUDENT ANSWER is relevant if it is concise and actually helps answer the QUESTION.

Start your reply with YES (relevant) or NO (not relevant), then give one short sentence explaining why."""


def relevance(inputs: dict, outputs: dict) -> dict:
    content = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    score, reply = ask_judge(RELEVANCE_INSTRUCTIONS, content)
    return {"key": "relevance", "score": score, "comment": reply}

In [61]:
RETRIEVAL_RELEVANCE_INSTRUCTIONS = """You are a teacher grading a quiz.
You will be given a QUESTION and some FACTS that were retrieved to answer it.
The FACTS are relevant if they contain any keywords or meaning related to the QUESTION.
They are NOT relevant if they are completely unrelated to the QUESTION.

Start your reply with YES (relevant) or NO (not relevant), then give one short sentence explaining why."""


def retrieval_relevance(inputs: dict, outputs: dict) -> dict:
    content = f"QUESTION: {inputs['question']}\n\nFACTS:\n{join_chunks(outputs['contexts'])}"
    score, reply = ask_judge(RETRIEVAL_RELEVANCE_INSTRUCTIONS, content)
    return {"key": "retrieval_relevance", "score": score, "comment": reply}

### 4.3 Test the evaluators on one answer before running everything
It's cheaper to catch a bug here than halfway through the experiment.

In [62]:
sample = examples[6]
sample_out = target(sample["inputs"])

for evaluator in [source_hit, answer_similarity]:
    print(evaluator(outputs=sample_out, reference_outputs=sample["outputs"]))
print(correctness(sample["inputs"], sample_out, sample["outputs"]))
print(groundedness(sample_out))
print(relevance(sample["inputs"], sample_out))
print(retrieval_relevance(sample["inputs"], sample_out))

{'key': 'source_hit', 'score': 1, 'comment': "expected First Principles of Gastroenterology.pdf | retrieved ['First Principles of Gastroenterology.pdf', 'First Principles of Gastroenterology.pdf', 'First Principles of Gastroenterology.pdf']"}
{'key': 'answer_similarity', 'score': 0.674}
{'key': 'correctness', 'score': 1, 'comment': "YES\nThe student answer accurately describes Charcot's triad in terms of the symptoms - jaundice, upper abdominal pain, and fever - which aligns with the correct definition provided for acute bacterial (ascending) cholangitis."}
{'key': 'groundedness', 'score': 1, 'comment': "YES\nThe student answer accurately describes Charcot's triad as consisting of jaundice, upper abdominal pain, and fever, which are all symptoms of acute cholangitis according to the provided facts."}
{'key': 'relevance', 'score': 1, 'comment': "YES (relevant)\nCharcot's triad specifically includes jaundice, upper abdominal pain, and fever, which are key indicators of acute cholangitis.

## 5. Run the evaluation (baseline experiment)

In [63]:
EVALUATORS = [source_hit, answer_similarity, correctness, groundedness, relevance, retrieval_relevance]

baseline = client.evaluate(
    target,
    data=DATASET_NAME,
    evaluators=EVALUATORS,
    experiment_prefix=f"groq-qwen-k{K}",
    metadata={"llm": GROQ_MODEL, "judge": GROQ_MODEL, "embeddings": "all-MiniLM-L6-v2", "k": K},
    max_concurrency=1,
)

View the evaluation results for experiment: 'groq-qwen-k3-3d4d28ff' at:
https://smith.langchain.com/o/8b966b74-a972-4bc3-a3db-0c647723e993/datasets/76d7fe4b-6c24-493c-ae7e-d6cf04f8d2c3/compare?selectedSessions=519e77bb-50a3-477d-b6f1-b6f1a98d9a59




13it [13:19, 61.49s/it]


## 6. Look at the results

In [64]:
import pandas as pd
pd.set_option("display.max_colwidth", 80)

df = baseline.to_pandas()

score_columns = [c for c in df.columns if c.startswith("feedback.")]
df[["inputs.question", "outputs.answer"] + score_columns]

,inputs.question,outputs.answer,feedback.source_hit,feedback.answer_similarity,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance
0,What is Charcot's triad?,"Charcot's triad refers to the combination of jaundice, upper abdominal pain,...",1.0,0.674,1,1,1,0
1,What is diabetes?,Diabetes is a chronic health condition characterized by high blood glucose l...,1.0,0.912,1,1,1,1
2,How is H. pylori infection treated?,The treatment of H. pylori infection involves eradicating the bacteria throu...,1.0,0.458,1,1,1,1
3,What are the types of poisonous snake bite?,The two types of poisonous snake bites are neurotoxic and vasculotoxic.,1.0,0.987,1,1,1,0
4,How long is the expiration dating period for 50% glycerin bulk allergen extr...,A total of six years from the time of extraction is allotted to 50% glycerin...,1.0,0.751,1,1,1,1
5,How should P. vivax malaria be treated?,P. vivax malaria should be treated with chloroquine for three days and prima...,1.0,0.731,1,1,1,1
6,How do I reset my Wi-Fi router?,"To reset your Wi-Fi router, follow these steps:\n\n1. Locate the router's ma...",NaN,-0.166,1,0,1,0
7,What color codes are used for allergen immunotherapy vials?,"For allergen immunotherapy vials, the standard color codes include:\n- RED f...",1.0,0.574,1,1,1,0
8,What does a complete clinical diagnosis include?,A complete clinical diagnosis includes reaching a clear understanding of the...,1.0,0.334,1,1,1,0
9,How many people in the United States had type 2 diabetes in 2007?,I do not have enough information to determine how many people in the United ...,1.0,0.890,0,0,0,0


In [65]:
df[score_columns].mean().round(2)

feedback.source_hit             1.00
feedback.answer_similarity      0.58
feedback.correctness            0.92
feedback.groundedness           0.77
feedback.relevance              0.85
feedback.retrieval_relevance    0.31
dtype: float64

In [66]:
df[df["feedback.correctness"] == 0][["inputs.question", "outputs.answer", "reference.answer"]]

,inputs.question,outputs.answer,reference.answer
9,How many people in the United States had type 2 diabetes in 2007?,I do not have enough information to determine how many people in the United ...,"In 2007, 23.6 million people, or 7.8% of the United States population, had t..."


**How to read the results.** The metrics point to different parts of the system:
- `source_hit` = 0 → a **retrieval** problem. The right PDF never reached the LLM, so the LLM had no chance.
- `source_hit` = 1 but `correctness` = 0 → a **generation** problem. The facts were there, but the model didn't use them well.
- `groundedness` = 0 → the model **made something up** that isn't in the chunks.
- Out-of-scope questions with `correctness` = 0 → the bot answered from its own knowledge instead of saying "I don't know".

Open the experiment link printed above to see every question, the retrieved chunks, and each judge's comment.

## 7. Experiment: change one thing and compare

Evaluation is useful because you can **compare**. Change one setting, run again with a new
`experiment_prefix`, then in LangSmith open the dataset → select both experiments → **Compare**.

Here we try retrieving 5 chunks instead of 3. Before running it, predict: which metrics should go up,
and which might go down?

In [67]:
K = 5
rag_chain = build_rag_chain(vector_store, llm, k=K)

experiment_k5 = client.evaluate(
    target,
    data=DATASET_NAME,
    evaluators=EVALUATORS,
    experiment_prefix=f"groq-qwen-k{K}",
    metadata={"llm": GROQ_MODEL, "judge": GROQ_MODEL, "embeddings": "all-MiniLM-L6-v2", "k": K},
    max_concurrency=1,
)

comparison = pd.DataFrame({
    "k=3": baseline.to_pandas()[score_columns].mean(),
    "k=5": experiment_k5.to_pandas()[score_columns].mean(),
}).round(2)
comparison

View the evaluation results for experiment: 'groq-qwen-k5-191dddd3' at:
https://smith.langchain.com/o/8b966b74-a972-4bc3-a3db-0c647723e993/datasets/76d7fe4b-6c24-493c-ae7e-d6cf04f8d2c3/compare?selectedSessions=5d9ba1af-5899-414c-9880-c7066526eecd




13it [27:17, 125.98s/it]


,k=3,k=5
feedback.source_hit,1.00,1.00
feedback.answer_similarity,0.58,0.60
feedback.correctness,0.92,1.00
feedback.groundedness,0.77,0.77
feedback.relevance,0.85,0.85
feedback.retrieval_relevance,0.31,0.23
